In [9]:
import pandas as pd
from openai import OpenAI
from autoddg import AutoDDG
from autoddg.utils import get_sample
from autoddg.evaluation import BaseEvaluator
from typing import Optional
from evaluate import load
# --- Import custom files ---
from prompts import ALL_RELATED_WORK_PROMPTS
from utils import log_result, run_description_experiment
import os 
import json
from cache_utils import run_with_caching, load_profile_from_cache#, MockAutoDDG           #Mock is for testing



In [10]:
# --- LLM Config ---
MODEL_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "api_key": "ollama",
    "model_name": "llama3.1:8b",
}

# Initialize Core Tools
client = OpenAI(api_key=MODEL_CONFIG["api_key"], base_url=MODEL_CONFIG["base_url"])


In [14]:
import json
import pandas as pd

results = pd.read_csv("results-updated.csv")
DATABASE_PATH_ = "../src/autoddg/database.json"

with open(DATABASE_PATH_, "r", encoding="utf-8") as f:
    db = json.load(f)

# dataset_name -> description lookup
name_to_desc = {
    v["dataset_name"].strip(): v.get("description", "")
    for v in db.values()
    if "dataset_name" in v
}

# add column (no overwriting other columns)
results["Reference_Description"] = (
    results["Dataset_Name"].astype(str).str.strip().map(name_to_desc)
)

# optional: keep blanks instead of NaN
results["Reference_Description"] = results["Reference_Description"].fillna("")

results.to_csv("results_refdesc.csv", index=False)


In [15]:
import pandas as pd
from enhanced_eval import evaluate_all


df = pd.read_csv("results_refdesc.csv")

#Compute metrics per row

output=[]

for _, row in df.iterrows():
    metrics = evaluate_all(
        row= row,
        client= client, 
        model_name= MODEL_CONFIG["model_name"] )
    # merge original row data + metrics into one dict
    row_and_metrics = {**row.to_dict(), **metrics}
    output.append(row_and_metrics)


metrics_df = pd.DataFrame(output)


metrics_df.to_csv("results-updated-eval.csv", index=False)

print("Enhanced evaluation saved to results-eval.csv")

⚠ WARNING: Could not parse JSON from LLM response — returning empty schema.
{'basic_info': {'domain_or_field': 'Immune Cell Analysis', 'primary_purpose': 'understanding the pathogenesis of myocarditis'}, 'data_characteristics': {'size_or_scale': None, 'data_format': None, 'data_types': ['numeric values', 'measures'], 'temporal_coverage': None, 'sample_unit': None}, 'provenance': {'collection_method': 'single-cell RNA sequencing and other techniques', 'data_source': None, 'collection_date': None, 'creators_or_curators': None, 'preprocessing_steps': None}, 'usage_context': {'typical_applications': 'understanding the mechanisms underlying myocarditis and develop more effective treatments', 'research_questions_addressed': 'mechanisms underlying myocarditis', 'how_used_in_paper': 'analyze the immune cells in patients with myocarditis who were undergoing ICI treatment', 'benchmark_or_evaluation_role': None}, 'quality_and_limitations': {'known_limitations': None, 'biases_or_caveats': None, 'q

In [ ]:
import pandas as pd, json
from enhanced_eval import extract_summary_profile_strict, extract_summary_profile_lenient

df = pd.read_csv("results_refdesc.csv")
row = df.iloc[2]  

profile = extract_summary_profile_lenient(row["Description_Text"], client, MODEL_CONFIG["model_name"])
print(json.dumps(profile, indent=2))

profile = extract_summary_profile_strict(row["Description_Text"], client, MODEL_CONFIG["model_name"])
print(json.dumps(profile, indent=2))